# Reflection Pattern

The first pattern we are going to implement is the **reflection pattern**.

This pattern allows the LLM to reflect and critique its outputs, following the next steps:

1. The LLM **generates** a candidate output. If you look at the diagram above, it happens inside the **"Generate"** box.
2. The LLM **reflects** on the previous output, suggesting modifications, deletions, improvements to the writing style, etc.
3. The LLM modifies the original output based on the reflections and another iteration begins ...

**Now, we are going to build, from scratch, each step, so that you can truly understand how this pattern works.**

## Generation Step

The first thing we need to consider is:

> What do we want to generate? A poem? An essay? Python code?

In this example, we'll test the Python coding skills of GPT-4o (remember that's the LLM we are going to use for all the tutorials). In particular, we are going to ask the LLM to code a famous sorting algorithm: **Merge Sort**.

---


### OpenAI Client and relevant imports

In [6]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display_markdown

load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")

client_gemini = OpenAI(
    api_key=gemini_api_key,  # Google Gemini API key
    base_url="https://generativelanguage.googleapis.com/v1beta/"  # Gemini base URL
)
model = "gemini-2.5-flash" # gemini-2.5-flash

We will start the **"generation"** chat history with the system prompt, as we said before. In this case, let the LLM act like a Python
programmer eager to receive feedback / critique by the user.

In [7]:
generation_chat_history = [
    {
        "role": "system",
        "content": "You are a Python programmer tasked with generating high quality Python code."
        "Your task is to Generate the best content possible for the user's request. If the user provides critique,"
        "respond with a revised version of your previous attempt."
    }
]

Now, as the user, we are going to ask the LLM to generate an implementation of the **Merge Sort** algorithm. Just add a new message with the **user** role to the chat history.

In [8]:
generation_chat_history.append(
    {
        "role": "user",
        "content": "Generate a Python implementation of the Merge Sort algorithm"
    }
)

Let's generate the first version of merge sort implementation.

In [9]:
generation_chat_history

[{'role': 'system',
  'content': "You are a Python programmer tasked with generating high quality Python code.Your task is to Generate the best content possible for the user's request. If the user provides critique,respond with a revised version of your previous attempt."},
 {'role': 'user',
  'content': 'Generate a Python implementation of the Merge Sort algorithm'}]

In [10]:
mergesort_code = client_gemini.chat.completions.create(
    messages=generation_chat_history,
    model=model
).choices[0].message.content

generation_chat_history.append(
    {
        "role": "assistant",
        "content": mergesort_code
    }
)

In [11]:
display_markdown(mergesort_code, raw=True)

Here's a Python implementation of the Merge Sort algorithm. Merge Sort is a highly efficient, comparison-based sorting algorithm. Most implementations produce a stable sort, meaning that the relative order of equal elements is preserved. It's a "divide and conquer" algorithm.

```python
def merge_sort(arr):
    """
    Sorts a list of comparable elements using the Merge Sort algorithm.

    Merge Sort is a divide and conquer algorithm that recursively divides
    the list into halves until it gets to lists of 0 or 1 element, then
    merges those halves back together in sorted order.

    Args:
        arr (list): The list of elements to be sorted.

    Returns:
        list: A new list containing the sorted elements.
              The original list 'arr' is not modified.
    """
    # Base case: A list of 0 or 1 element is already sorted.
    if len(arr) <= 1:
        return arr

    # Step 1: Divide the list into two halves
    mid = len(arr) // 2
    left_half = arr[:mid]
    right_half = arr[mid:]

    # Step 2: Recursively sort each half
    left_sorted = merge_sort(left_half)
    right_sorted = merge_sort(right_half)

    # Step 3: Merge the sorted halves back together
    return _merge(left_sorted, right_sorted)

def _merge(left, right):
    """
    Merges two sorted lists into a single sorted list.

    This is a helper function for merge_sort. It assumes that both 'left'
    and 'right' lists are already sorted.

    Args:
        left (list): The first sorted list.
        right (list): The second sorted list.

    Returns:
        list: A new list containing all elements from 'left' and 'right'
              in sorted order.
    """
    merged_list = []
    i = 0  # Pointer for the left list
    j = 0  # Pointer for the right list

    # Compare elements from both lists and append the smaller one
    # until one of the lists is exhausted.
    while i < len(left) and j < len(right):
        if left[i] <= right[j]: # Use <= for stable sort
            merged_list.append(left[i])
            i += 1
        else:
            merged_list.append(right[j])
            j += 1

    # Append any remaining elements from the left list (if any)
    while i < len(left):
        merged_list.append(left[i])
        i += 1

    # Append any remaining elements from the right list (if any)
    while j < len(right):
        merged_list.append(right[j])
        j += 1

    return merged_list

# --- Example Usage ---
if __name__ == "__main__":
    test_cases = [
        [],
        [1],
        [5, 2, 9, 1, 7],
        [3, 1, 4, 1, 5, 9, 2, 6],
        [10, 9, 8, 7, 6, 5, 4, 3, 2, 1], # Reverse sorted
        [1, 2, 3, 4, 5, 6, 7, 8, 9, 10], # Already sorted
        [5, 5, 5, 5, 5],                 # All equal elements
        ['banana', 'apple', 'zebra', 'grape', 'kiwi'] # Strings
    ]

    for i, arr in enumerate(test_cases):
        print(f"Test Case {i+1}:")
        print(f"Original: {arr}")
        sorted_arr = merge_sort(arr)
        print(f"Sorted:   {sorted_arr}\n")

    # Demonstrate stability with tuples (value, original_index)
    print("Demonstrating stability:")
    data = [(4, 'a'), (2, 'b'), (5, 'c'), (2, 'd'), (4, 'e')]
    print(f"Original: {data}")
    # Sort primarily by the numeric value, secondarily by original order for equal values
    sorted_data = merge_sort(data)
    print(f"Sorted:   {sorted_data}")
    # Expected output for stability: [(2, 'b'), (2, 'd'), (4, 'a'), (4, 'e'), (5, 'c')]
    # If it wasn't stable, (2, 'd') could come before (2, 'b'), or (4, 'e') before (4, 'a').
```

### Explanation:

1.  **`merge_sort(arr)` Function:**
    *   **Base Case:** If the input list `arr` has 0 or 1 element, it's considered sorted by definition. The function simply returns `arr`. This is crucial for stopping the recursion.
    *   **Divide:** The list `arr` is split into two halves: `left_half` and `right_half`. The `mid` index is calculated using integer division (`//`) to handle both even and odd length lists correctly.
    *   **Conquer (Recurse):** The `merge_sort` function is called recursively on `left_half` and `right_half`. This continues until the base case is reached (lists of size 0 or 1).
    *   **Combine (Merge):** Once the recursive calls return (meaning the `left_half` and `right_half` are now sorted), the `_merge` helper function is called to combine these two sorted halves into a single, fully sorted list.

2.  **`_merge(left, right)` Helper Function:**
    *   This function takes two already sorted lists, `left` and `right`.
    *   It initializes an empty list `merged_list` to store the combined sorted elements.
    *   Two pointers, `i` and `j`, are used to keep track of the current element being considered in `left` and `right` lists, respectively.
    *   **Comparison Loop:** The `while` loop runs as long as there are elements remaining in both `left` and `right` lists.
        *   It compares `left[i]` and `right[j]`.
        *   The smaller element is appended to `merged_list`, and its corresponding pointer (`i` or `j`) is incremented.
        *   Using `<=` instead of `<` ensures that if elements are equal, the element from the `left` list is picked first, which is key for maintaining **stability** in the sort.
    *   **Append Remaining Elements:** After the main loop finishes, one of the lists might still have remaining elements (e.g., if one list was much longer than the other). The two subsequent `while` loops simply append any remaining elements from `left` or `right` to `merged_list`. Since these remaining elements are already sorted within their respective lists, they can be appended directly without further comparison.
    *   Finally, the `merged_list` is returned.

### Time and Space Complexity:

*   **Time Complexity: O(n log n)**
    *   **Divide:** Dividing the list takes O(log n) steps because you repeatedly halve the list until you reach single elements.
    *   **Merge:** Merging two sorted lists of total size `n` takes O(n) time, as you iterate through all elements once.
    *   Since the merge step occurs at each level of the `log n` divisions, the total time complexity is `O(n log n)`. This holds true for best, average, and worst-case scenarios, making Merge Sort very reliable.

*   **Space Complexity: O(n)**
    *   Merge Sort requires extra space to store the merged sublists. In the worst case, you might need space proportional to the original list size (`n`) for the temporary lists created during the merging process.
    *   This is a trade-off for its consistent `O(n log n)` time complexity.

## Reflection Step

Now, let's allow the LLM to reflect on its outputs by defining another system prompt. This system prompt will tell the LLM to act as Andrej Karpathy, computer scientist and Deep Learning wizard.

>To be honest, I don't think the fact of acting like Andrej Karpathy will influence the LLM outputs, but it was fun :)


In [10]:
reflection_chat_history = [
    {
    "role": "system",
    "content": "You are Andrej Karpathy, an experienced computer scientist. You are tasked with generating critique and recommendations for the user's code",
    }
]

The user message, in this case,  is the essay generated in the previous step. We simply add the `mergesort_code` to the `reflection_chat_history`.

In [11]:
reflection_chat_history.append(
    {
        "role": "user",
        "content": mergesort_code
    }
)

Now, let's generate a critique to the Python code.

In [12]:
critique = client_gemini.chat.completions.create(
    messages=reflection_chat_history,
    model=model
).choices[0].message.content

In [13]:
display_markdown(critique, raw=True)

Okay, the provided code represents a good, functional implementation of the Merge Sort algorithm in Python. Here's a breakdown of its strengths, potential areas for minor improvement, and some higher-level considerations:

**Strengths:**

*   **Correctness:** The algorithm is correctly implemented and produces the expected sorted output. This is the most important aspect.
*   **Readability:** The code is well-structured and easy to follow. The use of meaningful variable names (`left_half`, `right_index`, `merged`) and clear comments enhances readability.
*   **Modularity:** The code is divided into two functions, `merge_sort` and `merge`, which promotes modularity and reusability.  This separation of concerns makes the code easier to understand, test, and maintain.
*   **Docstrings:** The docstrings are well-written and provide a clear explanation of the purpose, arguments, and return value of each function. This is crucial for documentation and maintainability.
*   **Base Case:** The base case in `merge_sort` (`if len(arr) <= 1: return arr`) is essential for stopping the recursion and ensuring that the algorithm terminates correctly.
*   **Clear Logic:** The logic within the `merge` function is straightforward and easy to understand. The use of `while` loops and index variables makes the merging process clear.

**Minor Potential Improvements (Mostly stylistic):**

*   **In-place Merge (Advanced):** The current implementation creates new lists in each recursive call and during the merge process.  While this simplifies the code, it can be less memory-efficient for very large lists. A more advanced (and complex) implementation could perform the merge in-place, modifying the original array directly to reduce memory overhead.  However, this increases the complexity and can make the code harder to read.  For most use cases, the current implementation is perfectly acceptable.
*   **Slightly More Concise Merge:** The `merged.extend()` calls could be slightly simplified using a conditional expression, though it's arguably less readable:

    ```python
    merged.extend(left[left_index:])
    merged.extend(right[right_index:])
    ```

    could become

    ```python
    merged.extend(left[left_index:] if left_index < len(left) else right[right_index:])
    merged.extend(right[right_index:] if right_index < len(right) else left[left_index:])
    ```
    (but I would argue against this, keep it as is)

*   **Type Hints (Modern Python):** For enhanced code clarity and static analysis, consider adding type hints:

    ```python
    from typing import List

    def merge_sort(arr: List[int]) -> List[int]:
        ...

    def merge(left: List[int], right: List[int]) -> List[int]:
        ...
    ```

**Higher-Level Considerations and Alternatives:**

*   **Python's Built-in `sorted()`:** Python has a built-in `sorted()` function, which is highly optimized and generally the best choice for sorting in most cases. It uses Timsort, a hybrid sorting algorithm derived from merge sort and insertion sort, optimized for real-world data.  The main reason to implement Merge Sort yourself is for educational purposes or when you need a specific sorting algorithm (e.g., for a coding interview or when you have very specific performance requirements).  Using `sorted()` is usually simpler and faster.

    ```python
    my_list = [38, 27, 43, 3, 9, 82, 10]
    sorted_list = sorted(my_list)
    print(f"Sorted list: {sorted_list}")
    ```

*   **Other Sorting Algorithms:** While Merge Sort has a guaranteed O(n log n) time complexity, other sorting algorithms might be more efficient in specific situations. For example:

    *   **Insertion Sort:** Can be very efficient for nearly sorted lists.
    *   **Quicksort:**  Generally faster than Merge Sort in practice, but has a worst-case time complexity of O(n^2).
    *   **Radix Sort:**  Can be very efficient for sorting integers or strings with a limited range.

    The choice of sorting algorithm depends on the characteristics of the data being sorted and the specific performance requirements of the application.

*   **Iterative Merge Sort:** While Merge Sort is commonly implemented recursively, it can also be implemented iteratively. The iterative version can sometimes be more memory-efficient because it avoids the overhead of recursive function calls.

**Summary:**

The provided code is a well-written and correct implementation of the Merge Sort algorithm. It's readable, modular, and well-documented. The potential areas for minor improvement are mostly stylistic and may not be necessary in all cases.  Consider using Python's built-in `sorted()` function for most practical sorting tasks unless you have a specific reason to implement Merge Sort yourself. Overall, great job!


Finally, we just need to add this *critique* to the `generation_chat_history`, in this case, as the `user` role.

In [14]:
generation_chat_history.append(
    {
        "role": "user",
        "content": critique
    }
)

In [15]:
generation_chat_history

[{'role': 'system',
  'content': "You are a Python programmer tasked with generating high quality Python code.Your task is to Generate the best content possible for the user's request. If the user provides critique,respond with a revised version of your previous attempt."},
 {'role': 'user',
  'content': 'Generate a Python implementation of the Merge Sort algorithm'},
 {'role': 'assistant',
  'content': '```python\ndef merge_sort(arr):\n  """\n  Sorts a list using the Merge Sort algorithm.\n\n  Args:\n    arr: The list to be sorted.\n\n  Returns:\n    A new list containing the sorted elements of the input list.\n  """\n\n  if len(arr) <= 1:\n    return arr  # Base case: already sorted\n\n  # 1. Divide the list into two halves\n  mid = len(arr) // 2\n  left_half = arr[:mid]\n  right_half = arr[mid:]\n\n  # 2. Recursively sort each half\n  left_half = merge_sort(left_half)\n  right_half = merge_sort(right_half)\n\n  # 3. Merge the sorted halves\n  return merge(left_half, right_half)\n\n\

## Next Steps

In [16]:
essay = client_gemini.chat.completions.create(
    messages=generation_chat_history,
    model=model
).choices[0].message.content

In [17]:
display_markdown(essay, raw=True)

```python
from typing import List

def merge_sort(arr: List[int]) -> List[int]:
  """
  Sorts a list using the Merge Sort algorithm.

  Args:
    arr: The list to be sorted.

  Returns:
    A new list containing the sorted elements of the input list.
  """

  if len(arr) <= 1:
    return arr  # Base case: already sorted

  # 1. Divide the list into two halves
  mid = len(arr) // 2
  left_half = arr[:mid]
  right_half = arr[mid:]

  # 2. Recursively sort each half
  left_half = merge_sort(left_half)
  right_half = merge_sort(right_half)

  # 3. Merge the sorted halves
  return merge(left_half, right_half)


def merge(left: List[int], right: List[int]) -> List[int]:
  """
  Merges two sorted lists into a single sorted list.

  Args:
    left: The first sorted list.
    right: The second sorted list.

  Returns:
    A new list containing all elements from both input lists, sorted.
  """

  merged = []
  left_index = 0
  right_index = 0

  # Compare elements from both lists and add the smaller one to the merged list
  while left_index < len(left) and right_index < len(right):
    if left[left_index] <= right[right_index]:
      merged.append(left[left_index])
      left_index += 1
    else:
      merged.append(right[right_index])
      right_index += 1

  # Add any remaining elements from the left list
  merged.extend(left[left_index:])

  # Add any remaining elements from the right list
  merged.extend(right[right_index:])

  return merged


# Example usage:
if __name__ == "__main__":
  my_list = [38, 27, 43, 3, 9, 82, 10]
  sorted_list = merge_sort(my_list)
  print(f"Original list: {my_list}")
  print(f"Sorted list: {sorted_list}")

  #Demonstrates use of built in sorted function
  my_list2 = [5,2,9,1,5,6]
  sorted_list2 = sorted(my_list2)
  print(f"Original list: {my_list2}")
  print(f"Sorted list using sorted(): {sorted_list2}")
```

The changes made incorporate the suggested type hints for increased clarity and static analysis capabilities. I've also added an example in the `if __name__ == "__main__":` block to explicitly demonstrate the usage of Python's built-in `sorted()` function, highlighting it as a potentially simpler and more efficient alternative for general sorting tasks.  No other changes were deemed necessary, as the original code was already well-structured and efficient.


And the iteration starts again ...

After **Generation Step (II)** the corrected Python code will be received, once again, by Karpathy. Then, the LLM will reflect on the corrected output, suggesting further improvements and the loop will go, over and over for a number **n** of total iterations.

> There's another possibility. Suppose the Reflection step can't find any further improvement. In this case, we can tell the LLM to output some stop string, like "OK" or "Good" that means the process can be stopped. However, we are going to follow the first approach, that is, iterating for a fixed number of times.

## Building a Reflection Agent

In [19]:
# @title
"""
This is a collection of helper functions and methods we are going to use in
the Agent implementation. You don't need to know the specific implementation
of these to follow the Agent code. But, if you are curious, feel free to check
them out.
"""

import time

from colorama import Fore
from colorama import Style


def completions_create(client, messages: list, model: str) -> str:
    """
    Sends a request to the client's `completions.create` method to interact with the language model.

    Args:
        client (OpenAI): The OpenAI client object
        messages (list[dict]): A list of message objects containing chat history for the model.
        model (str): The model to use for generating tool calls and responses.

    Returns:
        str: The content of the model's response.
    """
    response = client.chat.completions.create(messages=messages, model=model)
    return str(response.choices[0].message.content)


def build_prompt_structure(prompt: str, role: str, tag: str = "") -> dict:
    """
    Builds a structured prompt that includes the role and content.

    Args:
        prompt (str): The actual content of the prompt.
        role (str): The role of the speaker (e.g., user, assistant).

    Returns:
        dict: A dictionary representing the structured prompt.
    """
    if tag:
        prompt = f"<{tag}>{prompt}</{tag}>"
    return {"role": role, "content": prompt}

def update_chat_history(history: list, msg: str, role: str):
    """
    Updates the chat history by appending the latest response.

    Args:
        history (list): The list representing the current chat history.
        msg (str): The message to append.
        role (str): The role type (e.g. 'user', 'assistant', 'system')
    """
    history.append(build_prompt_structure(prompt=msg, role=role))


class ChatHistory(list):
    def __init__(self, messages: list | None = None, total_length: int = -1):
        """Initialise the queue with a fixed total length.

        Args:
            messages (list | None): A list of initial messages
            total_length (int): The maximum number of messages the chat history can hold.
        """
        if messages is None:
            messages = []

        super().__init__(messages)
        self.total_length = total_length

    def append(self, msg: str):
        """Add a message to the queue.

        Args:
            msg (str): The message to be added to the queue
        """
        if len(self) == self.total_length:
            self.pop(0)
        super().append(msg)



class FixedFirstChatHistory(ChatHistory):
    def __init__(self, messages: list | None = None, total_length: int = -1):
        """Initialise the queue with a fixed total length.

        Args:
            messages (list | None): A list of initial messages
            total_length (int): The maximum number of messages the chat history can hold.
        """
        super().__init__(messages, total_length)

    def append(self, msg: str):
        """Add a message to the queue. The first messaage will always stay fixed.

        Args:
            msg (str): The message to be added to the queue
        """
        if len(self) == self.total_length:
            self.pop(1)
        super().append(msg)

def fancy_print(message: str) -> None:
    """
    Displays a fancy print message.

    Args:
        message (str): The message to display.
    """
    print(Style.BRIGHT + Fore.CYAN + f"\n{'=' * 50}")
    print(Fore.MAGENTA + f"{message}")
    print(Style.BRIGHT + Fore.CYAN + f"{'=' * 50}\n")
    time.sleep(0.5)


def fancy_step_tracker(step: int, total_steps: int) -> None:
    """
    Displays a fancy step tracker for each iteration of the generation-reflection loop.

    Args:
        step (int): The current step in the loop.
        total_steps (int): The total number of steps in the loop.
    """
    fancy_print(f"STEP {step + 1}/{total_steps}")

In [20]:
BASE_GENERATION_SYSTEM_PROMPT = """
Your task is to Generate the best content possible for the user's request.
If the user provides critique, respond with a revised version of your previous attempt.
You must always output the revised content.
"""

BASE_REFLECTION_SYSTEM_PROMPT = """
You are tasked with generating critique and recommendations to the user's generated content.
If the user content has something wrong or something to be improved, output a list of recommendations
and critiques.
"""

import os
from dotenv import load_dotenv
load_dotenv()

class ReflectionAgent:
    """
    A class that implements a Reflection Agent, which generates responses and reflects
    on them using the LLM to iteratively improve the interaction. The agent first generates
    responses based on provided prompts and then critiques them in a reflection step.

    Attributes:
        model (str): The model name used for generating and reflecting on responses.
        client (OpenAI): An instance of the OpenAI client to interact with the language model.
    """

    def __init__(self, model: str = "gemini-2.0-flash"):
        self.client = OpenAI(
            api_key=os.getenv("GEMINI_API_KEY"),  # Google Gemini API key
            base_url="https://generativelanguage.googleapis.com/v1beta/"  # Gemini base URL
        )
        self.model = model

    def _request_completion(
        self,
        history: list,
        verbose: int = 0,
        log_title: str = "COMPLETION",
        log_color: str = "",
    ):
        """
        A private method to request a completion from the OpenAI model.

        Args:
            history (list): A list of messages forming the conversation or reflection history.
            verbose (int, optional): The verbosity level. Defaults to 0 (no output).

        Returns:
            str: The model-generated response.
        """
        output = completions_create(self.client, history, self.model)

        if verbose > 0:
            print(log_color, f"\n\n{log_title}\n\n", output)

        return output

    def generate(self, generation_history: list, verbose: int = 0) -> str:
        """
        Generates a response based on the provided generation history using the model.

        Args:
            generation_history (list): A list of messages forming the conversation or generation history.
            verbose (int, optional): The verbosity level, controlling printed output. Defaults to 0.

        Returns:
            str: The generated response.
        """
        return self._request_completion(
            generation_history, verbose, log_title="GENERATION", log_color=Fore.BLUE
        )

    def reflect(self, reflection_history: list, verbose: int = 0) -> str:
        """
        Reflects on the generation history by generating a critique or feedback.

        Args:
            reflection_history (list): A list of messages forming the reflection history, typically based on
                                       the previous generation or interaction.
            verbose (int, optional): The verbosity level, controlling printed output. Defaults to 0.

        Returns:
            str: The critique or reflection response from the model.
        """
        return self._request_completion(
            reflection_history, verbose, log_title="REFLECTION", log_color=Fore.GREEN
        )

    def run(
        self,
        user_msg: str,
        generation_system_prompt: str = "",
        reflection_system_prompt: str = "",
        n_steps: int = 10,
        verbose: int = 0,
    ) -> str:
        """
        Runs the ReflectionAgent over multiple steps, alternating between generating a response
        and reflecting on it for the specified number of steps.

        Args:
            user_msg (str): The user message or query that initiates the interaction.
            generation_system_prompt (str, optional): The system prompt for guiding the generation process.
            reflection_system_prompt (str, optional): The system prompt for guiding the reflection process.
            n_steps (int, optional): The number of generate-reflect cycles to perform. Defaults to 3.
            verbose (int, optional): The verbosity level controlling printed output. Defaults to 0.

        Returns:
            str: The final generated response after all cycles are completed.
        """
        generation_system_prompt += BASE_GENERATION_SYSTEM_PROMPT
        reflection_system_prompt += BASE_REFLECTION_SYSTEM_PROMPT

        # Given the iterative nature of the Reflection Pattern, we might exhaust the LLM context (or
        # make it really slow). That's the reason I'm limitting the chat history to three messages.
        # The `FixedFirstChatHistory` is a very simple class, that creates a Queue that always keeps
        # fixed the first message. I thought this would be useful for maintaining the system prompt
        # in the chat history.
        generation_history = FixedFirstChatHistory(
            [
                build_prompt_structure(prompt=generation_system_prompt, role="system"),
                build_prompt_structure(prompt=user_msg, role="user"),
            ],
            total_length=3,
        )

        reflection_history = FixedFirstChatHistory(
            [build_prompt_structure(prompt=reflection_system_prompt, role="system")],
            total_length=3,
        )

        for step in range(n_steps):
            if verbose > 0:
                fancy_step_tracker(step, n_steps)

            # Generate the response
            generation = self.generate(generation_history, verbose=verbose)
            update_chat_history(generation_history, generation, "assistant")
            update_chat_history(reflection_history, generation, "user")

            # Reflect and critique the generation
            critique = self.reflect(reflection_history, verbose=verbose)

            update_chat_history(generation_history, critique, "user")
            update_chat_history(reflection_history, critique, "assistant")

        return generation


In [21]:
agent = ReflectionAgent()  # gemini-2.5-flash

In [22]:
generation_system_prompt = "You are a Python programmer tasked with generating high quality Python code"

reflection_system_prompt = "You are Andrej Karpathy, an experienced computer scientist"

user_msg = "Generate a Python implementation of the Merge Sort algorithm"

In [23]:
final_response = agent.run(
    user_msg=user_msg,
    generation_system_prompt=generation_system_prompt,
    reflection_system_prompt=reflection_system_prompt,
    n_steps=3,
    verbose=1,
)


STEP 1/3

 

GENERATION

 ```python
def merge_sort(arr):
    """
    Sorts a list using the Merge Sort algorithm.

    Args:
      arr: The list to be sorted.

    Returns:
      A new list containing the sorted elements of arr.
    """

    if len(arr) <= 1:
        return arr  # Base case: already sorted

    # Divide the list into two halves
    mid = len(arr) // 2
    left = arr[:mid]
    right = arr[mid:]

    # Recursively sort the two halves
    left_sorted = merge_sort(left)
    right_sorted = merge_sort(right)

    # Merge the sorted halves
    return merge(left_sorted, right_sorted)


def merge(left, right):
    """
    Merges two sorted lists into a single sorted list.

    Args:
      left: The first sorted list.
      right: The second sorted list.

    Returns:
      A new list containing all elements from both input lists, sorted.
    """

    merged = []
    i = 0  # Index for the left list
    j = 0  # Index for the right list

    while i < len(left) and j < len(righ

In [24]:
print(final_response)

Thank you for the positive feedback! I'm glad I could deliver a solution that meets your expectations. I appreciate the thorough review process, as it helped me refine the code and documentation to a higher standard.

